# 🧬 장기 실종아동 나이 변환(Age Progression) 시스템
## Google Drive (AIHUB Data) + Colab T4 GPU 올인원 파이프라인

구글 드라이브 `My Drive/AIHUB Data`에 저장해두신 **AI Hub 4대 한국인 특화 데이터셋**을 직접 연결하여,
별도의 다운로드 대기 시간 없이 **초고속 압축 해제 ➔ 256x256 안면 정렬 ➔ 3단계 커리큘럼 학습 ➔ 타임랩스 시각화**까지 원클릭으로 완료합니다.

---
### 📚 연동되는 4대 핵심 데이터셋:
1. **가족 관계가 알려진 얼굴 이미지 데이터**: 아동-친부모 Cross-Attention ($L_{Kinship}$) 유전 특징 융합
2. **안면 인식 에이징(aging) 이미지 데이터**: 한국인 생애주기별 노화 곡선 ($L_{Age}$) 학습
3. **안면 랜드마크 데이터**: 정밀 5-Point 안면 정렬 및 하관 골격 마진 크롭
4. **한국인 헤어스타일 이미지**: 성인기 연령/성별 헤어 질감 및 자연스러운 윤곽 합성

---
### 📌 런타임 유형 설정 확인:
상단 메뉴 **[런타임] ➔ [런타임 유형 변경] ➔ 하드웨어 가속기: [T4 GPU]** 로 설정되어 있는지 확인하세요.

In [ ]:
# [Step 1] GPU 가속기 (T4 16GB VRAM) 상태 확인
!nvidia-smi

In [ ]:
# [Step 2] Google Drive 마운트 (AIHUB Data 폴더 연동)
from google.colab import drive
import os

drive.mount('/content/drive')

# 사용자의 구글 드라이브 내 AIHUB Data 폴더 설정
GDRIVE_DIR = '/content/drive/MyDrive/AIHUB Data'
CHECKPOINT_DIR = os.path.join(GDRIVE_DIR, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print('[OK] Google Drive mounted successfully!')
print(f'[OK] Target AI Hub Data Directory: {GDRIVE_DIR}')

In [ ]:
# [Step 3] GitHub 저장소 최신 코드 동기화
%cd /content
!rm -rf kinship-age-progression
!git clone https://github.com/ggg2214/kinship-age-progression.git
%cd kinship-age-progression
!ls -la

In [ ]:
# [Step 4] 필수 패키지 설치
!pip install -q torch torchvision fastapi uvicorn python-multipart Pillow

In [ ]:
# [Step 5] Google Drive 내 AIHUB Data 파일 및 압축본 자동 탐색
import os

GDRIVE_DIR = '/content/drive/MyDrive/AIHUB Data'
print(f'=== [{GDRIVE_DIR} 저장 파일 목록 검사] ===')

if not os.path.exists(GDRIVE_DIR):
    print(f'[경고] {GDRIVE_DIR} 폴더를 찾을 수 없습니다. 구글 드라이브 내 폴더명을 확인해 주세요.')
else:
    files = os.listdir(GDRIVE_DIR)
    print(f'발견된 항목 수: {len(files)}개')
    for f in sorted(files)[:25]:
        fpath = os.path.join(GDRIVE_DIR, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024) if os.path.isfile(fpath) else 0
        print(f' - {f} ({size_mb:.1f} MB)' if size_mb > 0 else f' - [폴더] {f}')
    if len(files) > 25:
        print(f' ... 외 {len(files) - 25}개')

In [ ]:
# [Step 6] 고속 압축 해제 (Google Drive -> Colab 로컬 초고속 SSD)
# 최신 코드 자동 동기화
!git pull origin main || true
# Google Drive의 수만 개 파일 I/O 병목을 피하기 위해 Colab 로컬 디스크(/content/raw_unpacked)로 추출

GDRIVE_DIR = '/content/drive/MyDrive/AIHUB Data'
LOCAL_RAW_DIR = '/content/raw_unpacked'

!python age_progression/data/aihub_pipeline.py \
    --action extract \
    --raw-dir "{GDRIVE_DIR}" \
    --target-dir "{LOCAL_RAW_DIR}"

# 압축 파일이 없거나 이미 풀려있는 경우 Google Drive 원본 디렉토리를 직접 소스로 사용
if os.path.exists(LOCAL_RAW_DIR) and len(os.listdir(LOCAL_RAW_DIR)) > 0:
    print('[OK] Colab 로컬 초고속 SSD(/content/raw_unpacked)로 압축 해제 완료!')
    ACTIVE_RAW_DIR = LOCAL_RAW_DIR
else:
    print('[INFO] 압축 파일이 없거나 이미 폴더 형태이므로 Google Drive 원본 디렉토리를 사용합니다.')
    ACTIVE_RAW_DIR = GDRIVE_DIR

In [ ]:
# [Step 7] 초경량 256x256 얼굴 크롭, 정렬 및 가족 Triplet 매니페스트 자동 생성
ALIGNED_DIR = '/content/data_aihub_aligned'
MANIFEST_PATH = '/content/data_aihub_aligned/manifest_aihub.json'

!python age_progression/data/aihub_pipeline.py \
    --action preprocess \
    --raw-dir "{ACTIVE_RAW_DIR}" \
    --aligned-dir "{ALIGNED_DIR}" \
    --manifest "{MANIFEST_PATH}"

# 정제된 256x256 데이터셋을 단일 압축본으로 구글 드라이브에 안전 백업
!python age_progression/data/aihub_pipeline.py \
    --action package \
    --aligned-dir "{ALIGNED_DIR}" \
    --archive "/content/drive/MyDrive/AIHUB Data/korean_face_aligned_256.tar.gz"

print('[OK] 256x256 Aligned Dataset and Triplet Manifest ready!')

In [ ]:
# [Step 8] 3단계 점진적 커리큘럼 학습 (Curriculum Training) 가동
# 최신 코드 자동 동기화
!git pull origin main || true
# Phase 1: 기본 안면 복원 및 정체성 보존 (Reconstruction & ArcFace ID)
# Phase 2: 한국인 에이징 데이터 기반 노화 방향 벡터 (Age Loss)
# Phase 3: 가족 관계 데이터 기반 부모 유전 정보 Cross-Attention 융합 (Kinship Loss)

MANIFEST_PATH = '/content/data_aihub_aligned/manifest_aihub.json'
CHECKPOINT_SAVE = '/content/drive/MyDrive/AIHUB Data/checkpoints/age_progression_korean_best.pt'

# 매니페스트가 비어있을 경우 벤치마크 데이터(FG-NET) 자동 대체
if not os.path.exists(MANIFEST_PATH) or os.path.getsize(MANIFEST_PATH) < 100:
    print('[INFO] AI Hub manifest가 없어 FG-NET 실제 종단면 벤치마크 데이터를 준비합니다...')
    !python age_progression/data/download_fgnet.py
    MANIFEST_PATH = 'data_fgnet/manifest_fgnet.json'

!python age_progression/train_curriculum.py \
    --device cuda \
    --image-size 256 \
    --batch-size 4 \
    --epochs 5 \
    --manifest "{MANIFEST_PATH}" \
    --save-path "{CHECKPOINT_SAVE}"

print(f'[SUCCESS] Curriculum Training Completed! Checkpoint saved to: {CHECKPOINT_SAVE}')

In [ ]:
# [Step 9] 학습 완료된 한국인 모델로 성장 타임랩스(10~60세) 시각화 검증
CKPT_PATH = '/content/drive/MyDrive/AIHUB Data/checkpoints/age_progression_korean_best.pt'
if not os.path.exists(CKPT_PATH):
    CKPT_PATH = 'checkpoints/age_progression_curriculum.pt'

OUTPUT_TIMELAPSE = '/content/drive/MyDrive/AIHUB Data/korean_sample_timelapse.png'

!python age_progression/visualize_samples.py \
    --child public/assets/real_korean_child_06.jpg \
    --father public/assets/real_korean_father_38.jpg \
    --mother public/assets/real_korean_mother_36.jpg \
    --ages 10 20 30 40 50 60 \
    --source-age 6 \
    --checkpoint "{CKPT_PATH}" \
    --output "{OUTPUT_TIMELAPSE}"

from IPython.display import Image, display
if os.path.exists(OUTPUT_TIMELAPSE):
    display(Image(OUTPUT_TIMELAPSE))
    print(f'[OK] 타임랩스 결과 이미지가 구글 드라이브에 저장되었습니다: {OUTPUT_TIMELAPSE}')

In [ ]:
# [Step 10] 구글 드라이브 최종 저장 상태 점검
!ls -lh '/content/drive/MyDrive/AIHUB Data/checkpoints/'
!ls -lh '/content/drive/MyDrive/AIHUB Data/korean_face_aligned_256.tar.gz' || true
!ls -lh '/content/drive/MyDrive/AIHUB Data/korean_sample_timelapse.png' || true
print('[DONE] 모든 학습 결과물 및 정제 데이터가 Google Drive에 성공적으로 보관되었습니다!')